# Modeling and training: hidden dissatisfaction

This notebook is a **thin reporting interface** over the tested pipeline in `src/`. It follows the staged presentation style of the reference capstone without copying its random split or modeling logic.

Before running, complete the unchecked data steps in `docs/modeling-training-plan.md` and build `data/processed/modelling_orders.parquet`.


## 1. Experiment contract

- Prediction point: delivery confirmation.
- Cohort: on-time delivered orders with observed reviews.
- Target: review score 1–2.
- Split: delivery-time train → selection → calibration → final test.
- Primary metric: PR-AUC; final utility includes calibration and finite review budgets.


In [ ]:
from pathlib import Path
import json
import pandas as pd

from hidden_dissatisfaction.modeling.training import TrainingConfig, run_training

ROOT = Path("..").resolve()
DATA = ROOT / "data/processed/modelling_orders.parquet"
REPORT = ROOT / "reports/experiments/latest.json"


## 2. Train and evaluate

All preprocessing is fitted inside estimator pipelines. Candidate selection, calibration, threshold selection, final evaluation, and artifact writing are implemented in reusable source code.


In [ ]:
report = run_training(
    DATA,
    config=TrainingConfig.from_toml(ROOT / "configs/experiment.toml"),
    output_root=ROOT,
)
print("Selected model:", report["selected_model"])
print("Calibration:", report["calibration_method"])


## 3. Temporal cohort evidence


In [ ]:
pd.DataFrame(report["partitions"]).T


## 4. Model comparison and final metrics


In [ ]:
display(
    pd.Series(report["selection_average_precision"], name="validation PR-AUC")
      .sort_values(ascending=False)
      .to_frame()
)
pd.Series(report["test_metrics"], name="test result").to_frame()


## 5. Finite-budget prioritization

`risk_recall` measures the share of low-review orders found. `risk_value_captured` is prioritization utility, not saved revenue or causal treatment impact.


In [ ]:
pd.DataFrame(report["budget_metrics"]).T


## 6. Audit outputs

The fitted model and row-level predictions are intentionally ignored by Git. The lightweight experiment report can be reviewed and versioned after a real run.


In [ ]:
print(ROOT / "artifacts/models/hidden_dissatisfaction.joblib")
print(ROOT / "artifacts/predictions/test_predictions.csv")
print(REPORT)


## Next analysis steps

Follow the remaining unchecked items in `docs/modeling-training-plan.md`: calibration/PR plots, temporal and subgroup error analysis, SHAP with non-causal wording, and the review-observation sensitivity model.
